In [1]:
import pandas as pd
from python_utilities.db_connection import DbConnection
import boto3
analytics_db = DbConnection('ANALYTICS', 'PROD_RDS')
# Create session with specific profile
session = boto3.Session(profile_name='739275445236_DataScienceUser')
s3 = session.client('s3')

INFO [2026-07-03 12:09:39] - PYTHON_UTILITIES - secret_utilities.py - get_db_secret_config - Credentials for database were read from secret.ini file
INFO [2026-07-03 12:09:39] - Found credentials in shared credentials file: ~/.aws/credentials


# Dataset Creation

In [2]:
query_true = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.model_name = 'invoice_detection_egvp'
AND lap.subtype = 'is_invoice_inside'
AND lap.value = "'True'"
AND lap.created_at >= '2026-06-01'
"""

invoice_df = analytics_db.sql_to_df(query_true)

In [3]:
query_false = """
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.model_name = 'invoice_detection_egvp'
AND lap.subtype = 'is_invoice_inside'
AND lap.value = "'False'"
AND lap.created_at >= '2026-06-01'
"""

not_invoice_df = analytics_db.sql_to_df(query_false)

In [4]:
invoice_df

,id,created_at,model_name,type,subtype,value,attachment_id
0,9887774,2026-06-01 06:28:54,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66979629
1,9887791,2026-06-01 06:28:55,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66979661
2,9969998,2026-06-01 22:48:44,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990795
3,9970015,2026-06-01 22:48:45,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990884
4,9970066,2026-06-01 22:48:49,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',66990893
...,...,...,...,...,...,...,...
5235,12203266,2026-07-03 09:55:15,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',70805539
5236,12203290,2026-07-03 09:55:17,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',70805552
5237,12203357,2026-07-03 09:55:22,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',70805568
5238,12203501,2026-07-03 09:55:32,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'True',70805619


In [5]:
not_invoice_df

,id,created_at,model_name,type,subtype,value,attachment_id
0,9885125,2026-06-01 02:07:18,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',66964484
1,9886360,2026-06-01 05:50:22,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',66977396
2,9887757,2026-06-01 06:28:53,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',66979093
3,9970032,2026-06-01 22:48:46,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',66990885
4,9970049,2026-06-01 22:48:48,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',66990879
...,...,...,...,...,...,...,...
8540,12203641,2026-07-03 09:55:44,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',70805615
8541,12203665,2026-07-03 09:55:46,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',70805665
8542,12203686,2026-07-03 09:55:47,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',70805203
8543,12203732,2026-07-03 09:55:51,invoice_detection_egvp,invoice_detection_egvp,is_invoice_inside,'False',70805410


In [6]:
attch_ids = invoice_df.attachment_id.unique().tolist() + not_invoice_df.attachment_id.unique().tolist()


In [15]:
import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')
from utils.prod_utils import pivot_attachment_predictions

In [8]:
all_predictions_query = f"""
SELECT *
FROM llm_attachments_predictions lap
WHERE lap.attachment_id IN ({','.join([str(id) for id in attch_ids])})
"""

all_predictions_df = analytics_db.sql_to_df(all_predictions_query)

In [9]:
pivoted_data = pivot_attachment_predictions(all_predictions_df)

In [10]:
pivoted_data.columns

Index(['attachment_id', 'drittauskunft_egvp_end_page',
       'drittauskunft_egvp_is_dritt', 'drittauskunft_egvp_is_invoice_inside',
       'drittauskunft_egvp_kein_ergebnis', 'drittauskunft_egvp_no_result',
       'drittauskunft_egvp_slug', 'drittauskunft_egvp_start_page',
       'egvp_standalone_invoice_invoice', 'egvp_standalone_invoice_reason',
       'egvp_standalone_invoice_slug', 'invoice_detection_egvp_end_page',
       'invoice_detection_egvp_is_invoice_inside',
       'invoice_detection_egvp_start_page', 'ladung_aftercourt_type',
       'ladung_class_pred', 'ladung_class_prob', 'ladung_debtor_name',
       'ladung_judicial_summon_date', 'ladung_slug', 'pfub_aftercourt_type',
       'pfub_class_pred', 'pfub_class_prob', 'pfub_creditor_name',
       'pfub_debtor_name', 'pfub_erlass_egvp_creditor_name',
       'pfub_erlass_egvp_debtor_name', 'pfub_erlass_egvp_is_invoice_inside',
       'pfub_erlass_egvp_is_pfub', 'pfub_erlass_egvp_slug', 'pfub_slug',
       'vermogenverzeichnis_

In [11]:
pivoted_data.ladung_class_pred.value_counts()

ladung_class_pred
False    11889
True      1896
Name: count, dtype: int64

In [ ]:
pivoted_data.pfub_erlass_egvp_is_pfub.value_counts()

In [ ]:
pivoted_data.vermogenverzeichnis_egvp_is_va.value_counts()

In [ ]:
pivoted_data.drittauskunft_egvp_is_dritt.value_counts()

In [ ]:
pivoted_data

In [ ]:
pivoted_data.invoice_detection_egvp_is_invoice_inside.value_counts()

#### Dataset 1 -> egvp invoice (will be used later)

In [ ]:
sample_pivot_invoice = pivoted_data[pivoted_data.invoice_detection_egvp_is_invoice_inside == "True"].sample(100, random_state=42)
sample_pivot_not_invoice = pivoted_data[pivoted_data.invoice_detection_egvp_is_invoice_inside == "False"].sample(100, random_state=42)

invoice_quality_analysis_df = pd.concat([sample_pivot_invoice, sample_pivot_not_invoice], axis=0)
invoice_quality_analysis_df.reset_index(drop=True, inplace=True)

import sys
sys.path.append('/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation')
from utils.prod_utils import get_data_by_attachment_id

# Fetch the textract text for each attachment and add it as a column.
attachment_texts = {}
for a_id in invoice_quality_analysis_df['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

invoice_quality_analysis_df['text'] = invoice_quality_analysis_df['attachment_id'].map(attachment_texts)
#invoice_quality_analysis_df.to_csv('invoice_quality_analysis_df.csv', index=False)


# Analysis Start

### 1) Dritt, Invoice=True 

In [ ]:
# Drittauskunft with invoice
mask = (pivoted_data.drittauskunft_egvp_is_dritt == "True") & (pivoted_data.invoice_detection_egvp_is_invoice_inside == "True") & (pivoted_data.vermogenverzeichnis_egvp_is_va == "False") & (pivoted_data.pfub_erlass_egvp_is_pfub == "False")
dritt_with_invoice = pivoted_data[mask]


In [ ]:
dritt_with_invoice

In [18]:
download_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/tmp_streamlit_show/pdf"
csv_dir = "/Users/melih.gorgulu/Desktop/Projects/aftercourt_automation/notebooks/invoice_detection/tmp_streamlit_show/csv"

In [ ]:
dritt_with_invoice_streamlit = dritt_with_invoice.sample(100, random_state=42)

# add text
attachment_texts = {}
for a_id in dritt_with_invoice_streamlit['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

dritt_with_invoice_streamlit['text'] = dritt_with_invoice_streamlit['attachment_id'].map(attachment_texts)

dritt_with_invoice_streamlit.to_csv(f'{csv_dir}/dritt_with_invoice_streamlit.csv', index=False)

### 2) VA, Invoice=True

In [ ]:
# vermogenverzeichnis with invoice
mask = (pivoted_data.drittauskunft_egvp_is_dritt == "False") & (pivoted_data.invoice_detection_egvp_is_invoice_inside == "True") & (pivoted_data.vermogenverzeichnis_egvp_is_va == "True") & (pivoted_data.pfub_erlass_egvp_is_pfub == "False")
va_with_invoice = pivoted_data[mask]

sample_size = min(100, len(va_with_invoice))
va_with_invoice_streamlit = va_with_invoice.sample(sample_size, random_state=42)

# add text
attachment_texts = {}
for a_id in va_with_invoice_streamlit['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

va_with_invoice_streamlit['text'] = va_with_invoice_streamlit['attachment_id'].map(attachment_texts)

va_with_invoice_streamlit.to_csv(f'{csv_dir}/va_with_invoice_streamlit.csv', index=False)


### 3) VA, DRITT AND INVOICE ALL TOGETHER

In [ ]:
# to have more data for this, i used bigger date range

In [12]:
# vermogenverzeichnis with invoice
mask = (pivoted_data.drittauskunft_egvp_is_dritt == "True") & (pivoted_data.invoice_detection_egvp_is_invoice_inside == "True") & (pivoted_data.vermogenverzeichnis_egvp_is_va == "True") & (pivoted_data.pfub_erlass_egvp_is_pfub == "False")
va_dritt_invoice = pivoted_data[mask]




In [20]:
sample_size = min(100, len(va_dritt_invoice))
va_dritt_invoice_streamlit = va_dritt_invoice.sample(sample_size, random_state=42)

# add text
attachment_texts = {}
for a_id in va_dritt_invoice_streamlit['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

va_dritt_invoice_streamlit['text'] = va_dritt_invoice_streamlit['attachment_id'].map(attachment_texts)

va_dritt_invoice_streamlit.to_csv(f'{csv_dir}/va_dritt_invoice_streamlit.csv', index=False)

### 4) ANNA's INVOICE MODEL != MELIH's INVOICE PAGE DETECTION MODEL


In [24]:
pivoted_data.columns

Index(['attachment_id', 'drittauskunft_egvp_end_page',
       'drittauskunft_egvp_is_dritt', 'drittauskunft_egvp_is_invoice_inside',
       'drittauskunft_egvp_kein_ergebnis', 'drittauskunft_egvp_no_result',
       'drittauskunft_egvp_slug', 'drittauskunft_egvp_start_page',
       'egvp_standalone_invoice_invoice', 'egvp_standalone_invoice_reason',
       'egvp_standalone_invoice_slug', 'invoice_detection_egvp_end_page',
       'invoice_detection_egvp_is_invoice_inside',
       'invoice_detection_egvp_start_page', 'ladung_aftercourt_type',
       'ladung_class_pred', 'ladung_class_prob', 'ladung_debtor_name',
       'ladung_judicial_summon_date', 'ladung_slug', 'pfub_aftercourt_type',
       'pfub_class_pred', 'pfub_class_prob', 'pfub_creditor_name',
       'pfub_debtor_name', 'pfub_erlass_egvp_creditor_name',
       'pfub_erlass_egvp_debtor_name', 'pfub_erlass_egvp_is_invoice_inside',
       'pfub_erlass_egvp_is_pfub', 'pfub_erlass_egvp_slug', 'pfub_slug',
       'vermogenverzeichnis_

In [31]:
mask =  (pivoted_data.invoice_detection_egvp_is_invoice_inside == "True") & (pivoted_data.egvp_standalone_invoice_invoice== "False")
melih_true_anna_false = pivoted_data[mask]

mask =  (pivoted_data.invoice_detection_egvp_is_invoice_inside == "False") & (pivoted_data.egvp_standalone_invoice_invoice== "True")
melih_false_anna_true = pivoted_data[mask]

In [32]:
sample_size = min(100, len(melih_true_anna_false))
melih_true_anna_false_streamlit = melih_true_anna_false.sample(sample_size, random_state=42)

# add text
attachment_texts = {}
for a_id in melih_true_anna_false_streamlit['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

melih_true_anna_false_streamlit['text'] = melih_true_anna_false_streamlit['attachment_id'].map(attachment_texts)

melih_true_anna_false_streamlit.to_csv(f'{csv_dir}/melih_true_anna_false_streamlit.csv', index=False)

Failed to fetch text for attachment 70454524: Query execution failed: 
        SELECT 
            llm_attachments.attachment_id,
            llm_attachments.s3_key as document_s3_key,
            llm_attachments.s3_bucket as document_s3_bucket,
            llm_attachments.file_name,
            llm_attachments_predictions.model_name,
            llm_attachments_predictions.type,
            llm_attachments_predictions.subtype,
            llm_attachments_predictions.value,
            textract_jobs.job_id AS textract_job_id,
            textract_jobs.status AS textract_status,
            textract_jobs.s3_link AS textract_s3_link
        FROM llm_attachments
        JOIN llm_attachments_predictions ON llm_attachments.attachment_id = llm_attachments_predictions.attachment_id
        JOIN textract_jobs ON llm_attachments.attachment_id = textract_jobs.attachment_id
        WHERE llm_attachments.attachment_id = '70454524'
    


In [35]:
melih_true_anna_false_streamlit.shape

(100, 43)

In [34]:
sample_size = min(100, len(melih_false_anna_true))
melih_false_anna_true_streamlit = melih_false_anna_true.sample(sample_size, random_state=42)

# add text
attachment_texts = {}
for a_id in melih_false_anna_true_streamlit['attachment_id'].unique():
    try:
        data = get_data_by_attachment_id(a_id, analytics_db, s3, verbose=False)
        attachment_texts[a_id] = data['text'].iloc[0] if not data.empty and 'text' in data.columns else None
    except Exception as e:
        print(f"Failed to fetch text for attachment {a_id}: {e}")
        attachment_texts[a_id] = None

melih_false_anna_true_streamlit['text'] = melih_false_anna_true_streamlit['attachment_id'].map(attachment_texts)

melih_false_anna_true_streamlit.to_csv(f'{csv_dir}/melih_false_anna_true_streamlit.csv', index=False)